# 04b — Focused U.S. EDA (optional, training data only)

This **adds** to `04_feature_engineering.ipynb`; it does not overwrite any completed work. Explore state/subcategory differences, duration, and funding goals using **U.S. training rows only**. Intervals describe uncertainty in *observed* rates, not causal effects. State-code suffixes remain a provisional U.S. geographic proxy.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HERE = Path.cwd()
PROJECT_ROOT = next((p for p in [HERE, *HERE.parents]
                     if (p / "utils" / "preprocessing.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run Jupyter from the repository root or a notebook subdirectory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from utils.features import rate_table

US = PROJECT_ROOT / "data" / "Processed-Data" / "us_only"
clean = pd.read_csv(US / "DSI_kickstarterscrape_dataset_cleaned.csv")
train_ids = pd.read_csv(US / "train_project_ids.csv")
training = clean.set_index("project id", drop=False).loc[train_ids["project id"]].copy().reset_index(drop=True)
training["target"] = train_ids["target"].astype(int).to_numpy()
assert np.array_equal(training["status"].eq("successful").astype(int), training["target"])
assert training["project id"].is_unique
print(f"U.S. training-only records: {len(training):,}; success rate: {training['target'].mean():.2%}")


U.S. training-only records: 26,945; success rate: 54.61%


In [2]:
# Wilson 95% intervals: show frequently represented states, not extreme tiny groups.
training["state"] = training["location"].astype("string").str.extract(
    r",\s*([A-Z]{2})\s*$", expand=False
)
state_rates = rate_table(training, "state", min_n=200).sort_values("n", ascending=False)
display(state_rates.head(20).round(3))
fig, ax = plt.subplots(figsize=(10, 5))
plot = state_rates.head(15).sort_values("rate")
ax.errorbar(plot["rate"], plot.index,
            xerr=[plot["rate"] - plot["ci_lo"], plot["ci_hi"] - plot["rate"]],
            fmt="o", capsize=2)
ax.axvline(training["target"].mean(), linestyle="--", color="gray", label="overall training rate")
ax.set_xlabel("Historical success rate (95% Wilson interval)")
ax.set_title("U.S. candidate training campaigns by state (15 largest groups)")
ax.legend()
plt.tight_layout()


,n,successes,rate,ci_lo,ci_hi,vs_base
state,,,,,,
CA,5772,3118,0.540,0.527,0.553,-0.006
NY,4627,3008,0.650,0.636,0.664,0.104
TX,1305,662,0.507,0.480,0.534,-0.039
IL,1214,726,0.598,0.570,0.625,0.052
FL,962,348,0.362,0.332,0.393,-0.184
WA,882,506,0.574,0.541,0.606,0.028
MA,852,548,0.643,0.610,0.675,0.097
PA,849,482,0.568,0.534,0.601,0.022
OR,802,480,0.599,0.564,0.632,0.052


In [3]:
# Subcategory comparisons: counts and intervals protect against tiny-sample conclusions.
subcat_rates = rate_table(training, "subcategory", min_n=120).sort_values("n", ascending=False)
display(subcat_rates.head(20).round(3))
print(f"Subcategories with at least 120 training campaigns: {len(subcat_rates)}")


,n,successes,rate,ci_lo,ci_hi,vs_base
subcategory,,,,,,
Short Film,2467,1521,0.617,0.597,0.636,0.070
Documentary,2172,1049,0.483,0.462,0.504,-0.063
Music,1973,1283,0.650,0.629,0.671,0.104
Theater,1531,1072,0.700,0.677,0.723,0.154
Film & Video,1512,664,0.439,0.414,0.464,-0.107
Indie Rock,1202,906,0.754,0.729,0.777,0.208
Rock,1108,754,0.681,0.652,0.707,0.134
Narrative Film,915,463,0.506,0.474,0.538,-0.040
Food,875,441,0.504,0.471,0.537,-0.042


Subcategories with at least 120 training campaigns: 45


In [4]:
# Duration/goal comparisons do not require unverified funded-date parsing.
length_bins = pd.cut(training["duration"], bins=[0, 15, 30, 45, 60, 90, float("inf")],
                     labels=["(0, 15] days", "(15, 30] days", "(30, 45] days",
                             "(45, 60] days", "(60, 90] days", ">90 days"])
display(rate_table(training, length_bins, min_n=30).round(3))

# Exploratory goal size RELATIVE to the category median.
# This is descriptive ONLY. For models, the shared FeatureBuilder learns medians
# inside each training fold, so validation labels/rows cannot influence medians.
log_goal = np.log1p(training["goal"].where(training["goal"].ge(0)))
category_medians = log_goal.groupby(training["category"]).transform("median")
relative_goal = log_goal - category_medians
relative_bins = pd.cut(relative_goal,
    bins=[-float("inf"), -1, -0.25, 0.25, 1, float("inf")],
    labels=["much below category median", "below median", "near median",
            "above median", "much above category median"])
display(rate_table(training, relative_bins, min_n=30).round(3))


,n,successes,rate,ci_lo,ci_hi,vs_base
duration,,,,,,
"(0, 15] days",1104,725,0.657,0.628,0.684,0.111
"(15, 30] days",10565,6032,0.571,0.561,0.580,0.025
"(30, 45] days",7659,4481,0.585,0.574,0.596,0.039
"(45, 60] days",4625,2197,0.475,0.461,0.489,-0.071
"(60, 90] days",2626,1165,0.444,0.425,0.463,-0.103
>90 days,366,116,0.317,0.271,0.366,-0.229


,n,successes,rate,ci_lo,ci_hi,vs_base
goal,,,,,,
much below category median,5524,3744,0.678,0.665,0.690,0.132
below median,5261,3262,0.620,0.607,0.633,0.074
near median,4979,2910,0.584,0.571,0.598,0.038
above median,6097,3103,0.509,0.496,0.521,-0.037
much above category median,5084,1697,0.334,0.321,0.347,-0.212


## Interpretation and next step

- These are **training-only associations**. Differences can reflect campaign mix and other confounders.
- The existing conservative pipeline already includes `goal_vs_cat`, `goal_vs_subcat`, `goal_per_day`, `duration`, and location/category features.
- `05a_safe_feature_check.ipynb` tests only a few **additional** candidate features, with all learned reference values fitted inside the CV fold.
- Do not enable title, reward, or reconstructed launch-date fields until historical availability has been verified. Do not use the test set for these comparisons.
